In [ ]:
# GroundingDINO
import io
import urllib.request

import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

MODEL_ID = "IDEA-Research/grounding-dino-base"  # Transformers version of groundingdino_swinb_cogcoor
# Same sample image as GroundingDINO/.asset/cat_dog.jpeg in the official repository
IMAGE_URL = "https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/.asset/cat_dog.jpeg"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID).to(device).eval()
TEXT_PROMPT = "cat. dog."  # Lowercase labels separated (and ended) by dots
BOX_THRESHOLD = 0.35
TEXT_THRESHOLD = 0.25

with urllib.request.urlopen(IMAGE_URL) as response:
    image = Image.open(io.BytesIO(response.read())).convert("RGB")

inputs = processor(images=image, text=TEXT_PROMPT, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
# Equivalent to box_threshold / text_threshold of groundingdino.util.inference.predict
results = processor.post_process_grounded_object_detection(
    outputs,
    inputs.input_ids,
    threshold=BOX_THRESHOLD,
    text_threshold=TEXT_THRESHOLD,
    target_sizes=[(image.height, image.width)],
)[0]
boxes = results["boxes"].cpu()        # (N, 4) xyxy in pixels (the official predict returns normalized cxcywh)
logits = results["scores"].cpu()      # (N,)
phrases = results["text_labels"]      # list of N phrases
print(f"boxes: {boxes}, logits: {logits}, phrases: {phrases}")
print(f"boxes_shape: {boxes.shape}, logits_shape: {logits.shape}, phrases_length: {len(phrases)}")

# Draw the detected boxes (equivalent to groundingdino.util.inference.annotate)
plt.figure(figsize=(10, 10))
plt.imshow(image)
for (x0, y0, x1, y1), score, phrase in zip(boxes.tolist(), logits.tolist(), phrases):
    plt.gca().add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, edgecolor="lime", facecolor="none", linewidth=2))
    plt.text(x0, y0, f"{phrase} {score:.2f}", color="black", backgroundcolor="lime", fontsize=10)
plt.axis('off')
plt.show()

In [ ]:
# SAM2
from contextlib import nullcontext

import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download
from transformers import Sam2Model, Sam2Processor

MODEL_ID = "facebook/sam2.1-hiera-large"
# Same sample image as sam2/notebooks/images/truck.jpg in the official repository
IMAGE_PATH = hf_hub_download(repo_id="hf-internal-testing/sam2-fixtures", filename="truck.jpg", repo_type="dataset")
device = "cuda" if torch.cuda.is_available() else "cpu"
autocast_context = (
    torch.autocast("cuda", dtype=torch.bfloat16)
    if device == "cuda"
    else nullcontext()
)

# Load the SAM2 model and processor
model = Sam2Model.from_pretrained(MODEL_ID).to(device).eval()
processor = Sam2Processor.from_pretrained(MODEL_ID)
print("Model loaded")
# Load the input image
image = Image.open(IMAGE_PATH).convert("RGB")

# Compute the image embeddings once and reuse them for each prompt (equivalent to SAM2ImagePredictor.set_image)
with torch.inference_mode(), autocast_context:
    pixel_values = processor(images=image, return_tensors="pt")["pixel_values"].to(device)
    image_embeddings = model.get_image_embeddings(pixel_values)


def predict(input_points=None, input_labels=None, input_boxes=None, multimask_output=True):
    """Equivalent to SAM2ImagePredictor.predict

    Args:
        input_points: Points with shape (num_points, 2) for a single object.
        input_labels: Labels with shape (num_points,). 1 is positive and 0 is negative.
        input_boxes: Boxes with shape (4,) for a single object or (num_boxes, 4) for batch inference.

    Returns:
        masks: Binary masks with shape (3, H, W), or (num_boxes, 3, H, W) for batch inference.
        scores: Predicted IoU scores with shape (3,), or (num_boxes, 3).
        logits: Low-resolution mask logits with shape (3, 256, 256), or (num_boxes, 3, 256, 256).
    """
    input_boxes = None if input_boxes is None else np.asarray(input_boxes)
    is_batch = input_boxes is not None and input_boxes.ndim == 2
    # Convert the prompts into the nested list format of the processor: [image, object, point, xy]
    inputs = processor(
        images=image,
        input_points=None if input_points is None else [[np.asarray(input_points).tolist()]],
        input_labels=None if input_labels is None else [[np.asarray(input_labels).tolist()]],
        input_boxes=None if input_boxes is None else [input_boxes.reshape(-1, 4).tolist()],
        return_tensors="pt",
    ).to(device)
    with torch.inference_mode(), autocast_context:
        outputs = model(
            image_embeddings=image_embeddings,
            input_points=inputs.get("input_points"),
            input_labels=inputs.get("input_labels"),
            input_boxes=inputs.get("input_boxes"),
            multimask_output=multimask_output,
        )
    # Upscale the masks to the original image size and binarize them
    masks = processor.post_process_masks(outputs.pred_masks.float().cpu(), inputs["original_sizes"].cpu())[0].numpy()
    scores = outputs.iou_scores[0].float().cpu().numpy()
    logits = outputs.pred_masks[0].float().cpu().numpy()
    if not is_batch:
        masks, scores, logits = masks[0], scores[0], logits[0]
    return masks, scores, logits


# Display the sorted masks, scores, and input points/boxes
def show_masks(image, masks, scores,
               input_points=None, input_labels=None, input_box=None,
               max_shown=3, fig_title=None):
    if (input_points is None) ^ (input_labels is None):
        raise ValueError("Both input_labels and input_points should be provided")
    if input_points is None:
        input_points = []
    if input_labels is None:
        input_labels = []
    if input_box is None:
        input_box = []
    # Sort masks by scores in descending order
    sorted_ind = np.argsort(scores)[::-1]
    masks = masks[sorted_ind]
    scores = scores[sorted_ind]
    num_masks_to_show = min(max_shown, len(sorted_ind))

    # Display each mask with the corresponding score and input points/boxes
    fig, axes = plt.subplots(1, num_masks_to_show, figsize=(num_masks_to_show * 5, 5))
    if num_masks_to_show == 1:
        axes = [axes]
    for i in range(num_masks_to_show):
        mask_overlay = np.zeros((*masks[i].shape, 4), dtype=np.float32)
        mask_overlay[masks[i] > 0] = [30 / 255, 144 / 255, 1.0, 0.5]
        axes[i].imshow(image)
        axes[i].imshow(mask_overlay)
        for input_point, input_label in zip(input_points, input_labels):
            axes[i].scatter(
                input_point[0],
                input_point[1],
                color="yellow" if input_label == 1 else "red",
                marker="*" if input_label == 1 else "o",
                s=250,
                edgecolor="black",
            )
        if len(input_box) == 4:
            rect = plt.Rectangle(
                (input_box[0], input_box[1]),
                input_box[2] - input_box[0],
                input_box[3] - input_box[1],
                linewidth=2,
                edgecolor="green",
                facecolor="none",
            )
            axes[i].add_patch(rect)
        axes[i].axis("off")
        axes[i].set_title(f"Mask {i+1}, Score: {scores[i]:.4f}")

    if fig_title is not None:
        fig.suptitle(fig_title)
        plt.tight_layout()
    plt.show()

In [ ]:
# Depth-Anything-3
import glob
import os
from pathlib import Path

import torch

from depth_anything_3.api import DepthAnything3

ROOT = Path.cwd().parent / "Depth-Anything-3"
MODEL_NAME = "DA3MONO-LARGE"
EXAMPLE_PATH = ROOT / "assets/examples/SOH"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = DepthAnything3.from_pretrained(f"depth-anything/{MODEL_NAME}")
model = model.to(device=device)

images = sorted(glob.glob(str(EXAMPLE_PATH / "*.png")))

# Multiple images inference
prediction = model.inference(
    images,  # List of image paths, PIL images or numpy arrays
)
# prediction.processed_images : [N, H, W, 3] uint8   array
print(prediction.processed_images.shape)
# prediction.depth            : [N, H, W]    float32 array
print(prediction.depth.shape)

# Metric and Monocular models do not output camera parameters
if MODEL_NAME not in ["DA3METRIC-LARGE", "DA3MONO-LARGE"]:
    # prediction.conf             : [N, H, W]    float32 array
    print(prediction.conf.shape)  
    # prediction.extrinsics       : [N, 3, 4]    float32 array # opencv w2c or colmap format
    print(prediction.extrinsics.shape)
    # prediction.intrinsics       : [N, 3, 3]    float32 array
    print(prediction.intrinsics.shape)

In [ ]:
# MapAnything
# Image-Only Inference
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
from pathlib import Path

# Get inference device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model - This requires internet access or the huggingface hub cache to be pre-downloaded
# For Apache 2.0 license model, use "facebook/map-anything-apache"
model = MapAnything.from_pretrained("facebook/map-anything-apache").to(device)

# Load and preprocess images from a folder or list of paths
images = ["/workspace/data/nuscenes/samples/CAM_FRONT/n008-2018-05-21-11-06-59-0400__CAM_FRONT__1526915250362465.jpg",
          "/workspace/data/nuscenes/samples/CAM_FRONT/n008-2018-05-21-11-06-59-0400__CAM_FRONT__1526915250862465.jpg",
          "/workspace/data/nuscenes/samples/CAM_FRONT/n008-2018-05-21-11-06-59-0400__CAM_FRONT__1526915251362465.jpg"]  # or ["path/to/img1.jpg", "path/to/img2.jpg", ...]
views = load_images(images)

# Run inference
predictions = model.infer(
    views,                            # Input views
    memory_efficient_inference=True,  # Trades off speed for more views (up to 2000 views on 140 GB). Trade off is negligible - see profiling section
    minibatch_size=None,              # Minibatch size for memory-efficient inference (use 1 for smallest GPU memory consumption). Default is dynamic computation based on available GPU memory.
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
    use_multiview_confidence=False,   # Enable multi-view depth consistency based confidence in place of learning-based one
)

# Access results for each view - Complete list of metric outputs
for i, pred in enumerate(predictions):
    # Geometry outputs
    pts3d = pred["pts3d"]                     # 3D points in world coordinates (B, H, W, 3)
    pts3d_cam = pred["pts3d_cam"]             # 3D points in camera coordinates (B, H, W, 3)
    depth_z = pred["depth_z"]                 # Z-depth in camera frame (B, H, W, 1)
    depth_along_ray = pred["depth_along_ray"] # Depth along ray in camera frame (B, H, W, 1)

    # Camera outputs
    ray_directions = pred["ray_directions"]   # Ray directions in camera frame (B, H, W, 3)
    intrinsics = pred["intrinsics"]           # Recovered pinhole camera intrinsics (B, 3, 3)
    camera_poses = pred["camera_poses"]       # OpenCV (+X - Right, +Y - Down, +Z - Forward) cam2world poses in world frame (B, 4, 4)
    cam_trans = pred["cam_trans"]             # OpenCV (+X - Right, +Y - Down, +Z - Forward) cam2world translation in world frame (B, 3)
    cam_quats = pred["cam_quats"]             # OpenCV (+X - Right, +Y - Down, +Z - Forward) cam2world quaternion in world frame (B, 4)

    # Quality and masking
    confidence = pred["conf"]                 # Per-pixel confidence scores (B, H, W)
    mask = pred["mask"]                       # Combined validity mask (B, H, W, 1)
    non_ambiguous_mask = pred["non_ambiguous_mask"]                # Non-ambiguous regions (B, H, W)
    non_ambiguous_mask_logits = pred["non_ambiguous_mask_logits"]  # Mask logits (B, H, W)

    # Scaling
    metric_scaling_factor = pred["metric_scaling_factor"]  # Applied metric scaling (B,)

    # Original input
    img_no_norm = pred["img_no_norm"]         # Denormalized input images for visualization (B, H, W, 3)

# Visualize the input images and the predicted depth maps (depth_z is metric depth in meters)
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(predictions), 2, figsize=(14, 4 * len(predictions)), squeeze=False)
for i, pred in enumerate(predictions):
    image = pred["img_no_norm"][0].cpu().numpy()            # (H, W, 3)
    depth = pred["depth_z"][0, ..., 0].float().cpu().numpy()  # (H, W) [m]
    valid = pred["mask"][0, ..., 0].cpu().numpy().astype(bool)
    depth_masked = np.where(valid, depth, np.nan)  # invalid pixels are shown as blank

    axes[i, 0].imshow(image)
    axes[i, 0].set_title(f"View {i}: input image")
    axes[i, 0].axis("off")
    im = axes[i, 1].imshow(depth_masked, cmap="turbo",
                           vmin=np.nanpercentile(depth_masked, 2), vmax=np.nanpercentile(depth_masked, 98))
    axes[i, 1].set_title(f"View {i}: depth_z [m] (scale factor {pred['metric_scaling_factor'][0].item():.2f})")
    axes[i, 1].axis("off")
    fig.colorbar(im, ax=axes[i, 1], fraction=0.03, label="depth [m]")
    print(f"View {i}: depth range {np.nanmin(depth_masked):.1f} - {np.nanmax(depth_masked):.1f} m, median {np.nanmedian(depth_masked):.1f} m")
plt.tight_layout()
plt.show()